<a href="https://colab.research.google.com/github/Mayank73636/Video_Content_Moderation_System_DL/blob/main/VCMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit transformers torch torchvision torchaudio
!pip install -q opencv-python-headless moviepy imageio ffmpeg-python
!pip install -q openai-whisper
!pip install -q better-profanity
!pip install -q pyngrok
!apt install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 8 not upgraded.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving VCMS.mp4 to VCMS (1).mp4


In [ ]:
import os

os.makedirs("frames", exist_ok=True)
os.makedirs("output", exist_ok=True)

In [ ]:
import cv2
import os

video_path = "VCMS.mp4"

cap = cv2.VideoCapture(video_path)

count = 0

while True:
    success, frame = cap.read()

    if not success:
        break

    frame_path = f"frames/frame_{count}.jpg"
    cv2.imwrite(frame_path, frame)

    count += 1

cap.release()

print("Frames Extracted:", count)

Frames Extracted: 802


In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image

# Load Pretrained AlexNet
model = models.alexnet(pretrained=True)

# Evaluation Mode
model.eval()

# Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
import os
import torch

unsafe_frames = []

frame_files = sorted(os.listdir("frames"))

for frame_name in frame_files:

    frame_path = os.path.join("frames", frame_name)

    try:
        image = Image.open(frame_path).convert("RGB")

        # Transform image
        img_tensor = transform(image)

        # Add batch dimension
        img_tensor = img_tensor.unsqueeze(0)

        # Prediction
        with torch.no_grad():
            outputs = model(img_tensor)

        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

        confidence, predicted_class = torch.max(probabilities, 0)

        confidence = confidence.item()

        print(frame_name, "-> Class:", predicted_class.item(),
              "Confidence:", confidence)

        # Threshold
        if confidence > 0.50:
            unsafe_frames.append({
                "frame": frame_name,
                "class_id": predicted_class.item(),
                "confidence": confidence
            })

    except Exception as e:
        print("Skipping:", frame_name, e)

print("\nUnsafe Frames Found:", len(unsafe_frames))

frame_0.jpg -> Class: 883 Confidence: 0.05587208643555641
frame_1.jpg -> Class: 883 Confidence: 0.054543036967515945
frame_10.jpg -> Class: 883 Confidence: 0.06873750686645508
frame_100.jpg -> Class: 601 Confidence: 0.0689711794257164
frame_101.jpg -> Class: 601 Confidence: 0.07838960736989975
frame_102.jpg -> Class: 601 Confidence: 0.06502979248762131
frame_103.jpg -> Class: 601 Confidence: 0.08209468424320221
frame_104.jpg -> Class: 601 Confidence: 0.0828782245516777
frame_105.jpg -> Class: 601 Confidence: 0.104713574051857
frame_106.jpg -> Class: 601 Confidence: 0.11248495429754257
frame_107.jpg -> Class: 601 Confidence: 0.12745198607444763
frame_108.jpg -> Class: 601 Confidence: 0.11655013263225555
frame_109.jpg -> Class: 601 Confidence: 0.1583670973777771
frame_11.jpg -> Class: 438 Confidence: 0.06860310584306717
frame_110.jpg -> Class: 601 Confidence: 0.1457372009754181
frame_111.jpg -> Class: 601 Confidence: 0.1492946892976761
frame_112.jpg -> Class: 601 Confidence: 0.1521968692

In [ ]:
import pandas as pd

df = pd.DataFrame(unsafe_frames)

report_path = "output/moderation_report.csv"

df.to_csv(report_path, index=False)

print("Report Saved:", report_path)

Report Saved: output/moderation_report.csv


In [ ]:
import ffmpeg

video_file = "VCMS (1).mp4"
audio_file = "output/audio.wav"

(
    ffmpeg
    .input(video_file)
    .output(audio_file)
    .run(overwrite_output=True)
)

print("Audio Extracted")

Audio Extracted


In [ ]:
import whisper

model = whisper.load_model("base")

result = model.transcribe("output/audio.wav")

transcript = result["text"]

print(transcript)

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 200MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 پைலے میری پாچھ سے ஆ کر روڑ کا சாப்தے اரை, அرام سے دیل کرتے ہیں ایسے عملی دیکھا نہیں کریں یہ گும کا ہے, اس نے کسے وہ لہ نہیں, وہ لہ واپس, وہ لہ அرام سے بات کیجے نہ امری دیکھا نہیں کیا ضرورتے اچھا, سندی, وہ لہ ایک لہ, سندی, وہ لہا ہے اس کو, کیوں, وہ لہکی ایک پر, وہ لہکتے ہیں دولوں, وہ لہکتے ہیں ایک نے, وہ لہک نہیں کریں آپ پاچھ زار کی توپیک پیونہ م your external آپ پائے کنت میں نہیں


In [ ]:
from better_profanity import profanity

contains_bad_words = profanity.contains_profanity(transcript)

if contains_bad_words:
    print("⚠ Toxic Language Detected")
else:
    print("✅ Clean Audio")

✅ Clean Audio


In [ ]:
with open("output/transcript.txt", "w") as f:
    f.write(transcript)

print("Transcript Saved")

Transcript Saved


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd

st.title("🎥 Video Content Moderation System")

st.subheader("Moderation Report")

df = pd.read_csv("output/moderation_report.csv")

st.dataframe(df)

st.subheader("Transcript")

with open("output/transcript.txt", "r") as f:
    transcript = f.read()

st.write(transcript)

Overwriting app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q -O - ipv4.icanhazip.com

!npm install -g localtunnel

34.12.236.159
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [ ]:
!lt --port 8501

your url is: https://wild-sloths-argue.loca.lt
